# Sustained Attention Benchmark

Evaluates whether models can track and aggregate specific targets accurately across a long sequential context, without skipping entries, approximating, or losing count midway. Runs across all models available in the Kaggle environment (`kbench.llms`). A fixed judge model (`kbench.judge_llm`) scores all responses to ensure consistent cross-model evaluation.

Each task row contains 5 judge-evaluated criteria plus 1 regex hard-check on the final numeric answer. Total assertions per model: 18 (6 per row).

> **Environment note:** Designed to run inside a Kaggle notebook where `kbench.llm`, `kbench.judge_llm`, and `kbench.llms` are auto-configured.

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

In [ ]:
TASK_DATA = pd.DataFrame([
    {
        "context": (
            "Transaction 1: Alice sent $200 to Bob. "
            "Transaction 2: Carol bought groceries for $45. "
            "Transaction 3: Bob transferred $80 to Dave. "
            "Transaction 4: Eve paid $120 for utilities. "
            "Transaction 5: Alice received $300 from Frank. "
            "Transaction 6: Dave spent $60 at a restaurant. "
            "Transaction 7: Carol received $150 from Grace. "
            "Transaction 8: Frank paid $90 for insurance. "
            "Transaction 9: Bob bought electronics for $400. "
            "Transaction 10: Eve received $250 from Alice. "
            "Transaction 11: Grace paid $30 for a subscription. "
            "Transaction 12: Dave received $110 from Carol. "
            "Transaction 13: Alice paid $75 for phone bills. "
            "Transaction 14: Frank spent $200 on travel. "
            "Transaction 15: Bob received $95 from Eve."
        ),
        "question": "What is Alice's net balance across all transactions? Show your working.",
        "expected_answer": "-.*225|negative.*225",
        "criteria": [
            "Response accounts for Alice sending $200 (debit).",
            "Response accounts for Alice receiving $300 from Frank (credit).",
            "Response accounts for Alice paying $75 for phone bills (debit).",
            "Response accounts for Eve receiving $250 from Alice (debit).",
            "Final net balance calculated is negative $225.",
        ],
    },
    {
        "context": (
            "Day 1: Temperature recorded at 18C, humidity 65%. "
            "Day 2: Temperature recorded at 21C, humidity 70%. "
            "Day 3: Temperature recorded at 17C, humidity 80%. "
            "Day 4: Temperature recorded at 23C, humidity 55%. "
            "Day 5: Temperature recorded at 19C, humidity 60%. "
            "Day 6: Temperature recorded at 25C, humidity 45%. "
            "Day 7: Temperature recorded at 22C, humidity 50%. "
            "Day 8: Temperature recorded at 16C, humidity 85%. "
            "Day 9: Temperature recorded at 20C, humidity 75%. "
            "Day 10: Temperature recorded at 24C, humidity 40%. "
            "Day 11: Temperature recorded at 18C, humidity 68%. "
            "Day 12: Temperature recorded at 21C, humidity 72%. "
            "Day 13: Temperature recorded at 26C, humidity 38%. "
            "Day 14: Temperature recorded at 15C, humidity 90%."
        ),
        "question": "How many days had both temperature above 20C and humidity below 60%?",
        "expected_answer": "3",
        "criteria": [
            "Response correctly identifies Day 6 as meeting both conditions (25C, 45%).",
            "Response correctly identifies Day 10 as meeting both conditions (24C, 40%).",
            "Response correctly identifies Day 13 as meeting both conditions (26C, 38%).",
            "Response correctly excludes days meeting only one condition.",
            "Final count given is 3 days.",
        ],
    },
    {
        "context": (
            "Item A: Category=Electronics, Price=$299, Stock=14 units. "
            "Item B: Category=Clothing, Price=$49, Stock=200 units. "
            "Item C: Category=Electronics, Price=$899, Stock=6 units. "
            "Item D: Category=Food, Price=$12, Stock=500 units. "
            "Item E: Category=Clothing, Price=$89, Stock=75 units. "
            "Item F: Category=Electronics, Price=$199, Stock=30 units. "
            "Item G: Category=Food, Price=$8, Stock=1000 units. "
            "Item H: Category=Clothing, Price=$129, Stock=40 units. "
            "Item I: Category=Electronics, Price=$649, Stock=9 units. "
            "Item J: Category=Food, Price=$22, Stock=300 units."
        ),
        "question": "What is the total inventory value of all Electronics items?",
        "expected_answer": "21[,]?391",
        "criteria": [
            "Response correctly calculates Item A value as $4186 (299 x 14).",
            "Response correctly calculates Item C value as $5394 (899 x 6).",
            "Response correctly calculates Item F value as $5970 (199 x 30).",
            "Response correctly calculates Item I value as $5841 (649 x 9).",
            "Total inventory value calculated is $21391.",
        ],
    },
])


@kbench.task(name="sustained_attention")
def sustained_attention(
    llm,
    context: str,
    question: str,
    expected_answer: str,
    criteria: list,
):
    """
    Evaluating sustained attention: model must track and aggregate specific
    targets accurately across a long sequential context. Judge is a fixed
    external model. Each row has 1 regex hard-check plus 5 judge criteria.
    """
    prompt = (
        f"Read the following records carefully and answer the question. "
        f"Track every relevant entry -- do not skip or approximate.\n\n"
        f"Records:\n{context}\n\n"
        f"Question: {question}"
    )

    response = llm.prompt(prompt)

    kbench.assertions.assert_contains_regex(
        pattern=expected_answer,
        text=response,
        expectation="Response should contain the correct numeric answer",
    )

    report = kbench.assertions.assess_response_with_judge(
        criteria=criteria,
        response_text=response,
        judge_llm=kbench.judge_llm,
    )
    for result in report.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=result.criterion,
        )

In [ ]:
all_runs = []

for model_name, model_llm in kbench.llms.items():
    print(f"\n--- {model_name} ---")
    for i, row in TASK_DATA.iterrows():
        run = sustained_attention.run(
            llm=model_llm,
            context=row["context"],
            question=row["question"],
            expected_answer=row["expected_answer"],
            criteria=row["criteria"],
        )
        all_runs.append((model_name, i, run))
        passed = sum(ar.passed for ar in run.assertion_results)
        total = len(run.assertion_results)
        print(f"  Row {i}: {run.status.name} -- {passed}/{total} assertions passed")

In [ ]:
def run_to_record(model_name, row_index, run):
    passed = sum(ar.passed for ar in run.assertion_results)
    total = len(run.assertion_results)
    assertion_summary = "; ".join(
        f"{'PASS' if ar.passed else 'FAIL'}: {ar.expectation}"
        for ar in run.assertion_results
    )
    return {
        "model": model_name,
        "row": row_index,
        "status": run.status.name,
        "assertions_passed": passed,
        "assertions_total": total,
        "pass_rate": round(passed / total, 2) if total > 0 else None,
        "error_message": run.error_message,
        "assertion_details": assertion_summary,
    }


records = [run_to_record(model_name, i, run) for model_name, i, run in all_runs]
results_df = pd.DataFrame(records)
pd.set_option("display.max_colwidth", 100)

print("\n=== Per-row results ===")
display(results_df)

print("\n=== Summary by model ===")
summary = (
    results_df.groupby("model")
    .agg(
        assertions_passed=("assertions_passed", "sum"),
        assertions_total=("assertions_total", "sum"),
    )
    .assign(pass_rate=lambda df: (df["assertions_passed"] / df["assertions_total"]).round(3))
    .sort_values("pass_rate", ascending=False)
    .reset_index()
)
display(summary)

In [ ]:
import plotly.graph_objects as go

display_summary = summary.copy()
display_summary["pass_rate"] = display_summary["pass_rate"].apply(lambda x: f"{x:.1%}")

fig = go.Figure(data=[go.Table(
    columnwidth=[4, 2, 2, 2],
    header=dict(
        values=["<b>Model</b>", "<b>Passed</b>", "<b>Total</b>", "<b>Pass Rate</b>"],
        fill_color="#2c5f8a",
        font=dict(color="white", size=12),
        align="left",
        height=32,
    ),
    cells=dict(
        values=[
            display_summary["model"],
            display_summary["assertions_passed"],
            display_summary["assertions_total"],
            display_summary["pass_rate"],
        ],
        fill_color=[["#f0f4f8" if i % 2 == 0 else "white" for i in range(len(display_summary))]],
        font=dict(color="#333333", size=11),
        align="left",
        height=26,
    ),
)])

fig.update_layout(
    title=dict(text="Sustained Attention: Pass Rate by Model", font=dict(size=14)),
    margin=dict(l=10, r=10, t=50, b=10),
    width=700,
    height=max(300, len(display_summary) * 28 + 100),
)

output_path = "sustained_attention_results.png"
fig.write_image(output_path, scale=2)
print(f"Saved: {output_path}")
fig.show()